# SensoriMotor Coupling interaction

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az 

# set random seed for reproducibility
random_seed = 42

In [21]:
df = pd.read_csv('data/amygdala_sensorymotor.csv')
df.head()

,sub,Condition,Event.Nr,CDA.AmpSum,expected_value,pe,scr,index,subject,trialNo,condition,coupling,amg,amg_vmpfc,amg_sm
0,sub-189,CSplusUS1,1,0.2852,0.801175,0.500000,0.2852,3036,sub-189,1,CSplusUS1,0.904762,0.476625,0.833333,-0.333333
1,sub-189,CSminus1,2,0.1033,0.796939,-0.500000,0.1033,3037,sub-189,2,CSminus1,-0.380952,0.081692,0.428571,0.119048
2,sub-189,CSplus1,3,0.0783,0.799047,-0.501304,0.0783,3038,sub-189,3,CSplus1,0.571429,-0.219659,0.690476,-0.047619
3,sub-189,CSplusUS1,4,0.1772,0.801165,0.500006,0.1772,3039,sub-189,4,CSplusUS1,0.619048,0.006618,0.880952,0.142857
4,sub-189,CSminus1,5,0.0000,0.794832,-0.498696,0.0000,3040,sub-189,5,CSminus1,0.833333,-0.188212,0.595238,0.642857


In [22]:
# add group 
df_group = pd.read_csv('data/scr_brain_group.csv')
df_group.head()

,sub,Condition,Event.Nr,CDA.AmpSum,pe,scr,index,subject,trialNo,condition,coupling,amg,amg_vmpfc,sub_id,group,Gender,Age,n_zero_scr_x,n_zero_scr_y
0,sub-189,CSplusUS1,1,0.2852,0.500000,0.2852,3036,sub-189,1,CSplusUS1,0.904762,0.476625,0.833333,sub-189,HC,2.0,26.0,15,15
1,sub-189,CSminus1,2,0.1033,-0.500000,0.1033,3037,sub-189,2,CSminus1,-0.380952,0.081692,0.428571,sub-189,HC,2.0,26.0,15,15
2,sub-189,CSplus1,3,0.0783,-0.750000,0.0783,3038,sub-189,3,CSplus1,0.571429,-0.219659,0.690476,sub-189,HC,2.0,26.0,15,15
3,sub-189,CSplusUS1,4,0.1772,0.686106,0.1772,3039,sub-189,4,CSplusUS1,0.619048,0.006618,0.880952,sub-189,HC,2.0,26.0,15,15
4,sub-189,CSminus1,5,0.0000,-0.250000,0.0000,3040,sub-189,5,CSminus1,0.833333,-0.188212,0.595238,sub-189,HC,2.0,26.0,15,15


In [24]:
df = pd.merge(df, df_group[['sub','group', 'trialNo']])
df.head()

,sub,Condition,Event.Nr,CDA.AmpSum,expected_value,pe,scr,index,subject,trialNo,condition,coupling,amg,amg_vmpfc,amg_sm,group
0,sub-189,CSplusUS1,1,0.2852,0.801175,0.500000,0.2852,3036,sub-189,1,CSplusUS1,0.904762,0.476625,0.833333,-0.333333,HC
1,sub-189,CSminus1,2,0.1033,0.796939,-0.500000,0.1033,3037,sub-189,2,CSminus1,-0.380952,0.081692,0.428571,0.119048,HC
2,sub-189,CSplus1,3,0.0783,0.799047,-0.501304,0.0783,3038,sub-189,3,CSplus1,0.571429,-0.219659,0.690476,-0.047619,HC
3,sub-189,CSplusUS1,4,0.1772,0.801165,0.500006,0.1772,3039,sub-189,4,CSplusUS1,0.619048,0.006618,0.880952,0.142857,HC
4,sub-189,CSminus1,5,0.0000,0.794832,-0.498696,0.0000,3040,sub-189,5,CSminus1,0.833333,-0.188212,0.595238,0.642857,HC


In [27]:
# Encode 'sub' as integer indices
df['sub_idx'] = pd.Categorical(df['sub']).codes
n_subs = df['sub_idx'].nunique()

# Encode 'group' as integer indices (make ordering explicit!)
# Data uses: HC (healthy controls), VCC (combat controls), VPTSD (PTSD)
group_order = ['HC', 'VCC', 'VPTSD']
df['group'] = pd.Categorical(df['group'], categories=group_order, ordered=True)
df['group_idx'] = df['group'].cat.codes
n_groups = df['group_idx'].nunique()

# Check which group is reference (index 0)
print("Group coding (0 = reference):", {g: i for i, g in enumerate(df['group'].cat.categories)})

# Extract variables
pe = df['pe'].values
coupling = df['amg_sm'].values
amg = df['amg'].values
trialNo = df['trialNo'].values
sub_idx = df['sub_idx'].values
group_idx = df['group_idx'].values

with pm.Model() as model:
    
    # Fixed effects (main effects)
    beta_coupling = pm.Normal('beta_coupling', mu=0, sigma=1)  # Coupling effect for reference group
    beta_amg = pm.Normal('beta_amg', mu=0, sigma=1)
    beta_trialNo = pm.Normal('beta_trialNo', mu=0, sigma=1)
    
    # Main effect of group (reference group = 0, so n_groups-1 parameters)
    beta_group_raw = pm.Normal('beta_group_raw', mu=0, sigma=1, shape=n_groups - 1)
    beta_group = pm.math.concatenate([[0], beta_group_raw])  # Pad reference group with 0
    
    # Group × Coupling interaction (deviation from reference group's slope)
    beta_interaction_raw = pm.Normal('beta_interaction_raw', mu=0, sigma=1, shape=n_groups - 1)
    beta_interaction = pm.math.concatenate([[0], beta_interaction_raw])  # Pad reference with 0
    
    # Hyperpriors for random intercepts
    mu_a = pm.Normal('mu_a', mu=0, sigma=1)
    sigma_a = pm.HalfNormal('sigma_a', sigma=1)
    
    # Non-centered random intercepts
    z_a = pm.Normal('z_a', mu=0, sigma=1, shape=n_subs)
    a = pm.Deterministic('a', mu_a + z_a * sigma_a)
    
    # Expected value of outcome
    mu = (
        a[sub_idx] +
        beta_group[group_idx] +                        # Main effect of group
        beta_coupling * coupling +                      # Main effect of coupling (reference slope)
        beta_interaction[group_idx] * coupling +        # Interaction: group-specific slope adjustment
        beta_amg * amg +
        beta_trialNo * trialNo
    )
    
    # Likelihood
    sigma = pm.HalfNormal('sigma', sigma=1)
    y_obs = pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
    
    trace = pm.sample(chains=4, random_seed=random_seed, return_inferencedata=True,
                      idata_kwargs={"log_likelihood": True})



Group coding (0 = reference): {'HC': 0, 'VCC': 1, 'VPTSD': 2}


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, beta_group_raw, beta_interaction_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 9 seconds.


In [ ]:



# %% grab group specific slopes for coupling
# group specific slope

# Post-hoc calculation of group-specific slopes
posterior = trace.posterior
slope_hipp_HC = posterior['beta_coupling']
if n_groups != 3:
    raise ValueError(f"Expected 3 groups {group_order}, but found n_groups={n_groups}.")
slope_hipp_VCC = posterior['beta_coupling'] + posterior['beta_interaction_raw'][:, :, group_order.index('VCC') - 1]
slope_hipp_VPTSD = posterior['beta_coupling'] + posterior['beta_interaction_raw'][:, :, group_order.index('VPTSD') - 1]

print("HC slope (mean, SD, HDI):", float(slope_hipp_HC.mean()), float(slope_hipp_HC.std()), az.hdi(slope_hipp_HC.values.flatten(), hdi_prob=0.89))
print("VCC slope (mean, SD, HDI):", float(slope_hipp_VCC.mean()), float(slope_hipp_VCC.std()), az.hdi(slope_hipp_VCC.values.flatten(), hdi_prob=0.89))
print("VPTSD slope (mean, SD, HDI):", float(slope_hipp_VPTSD.mean()), float(slope_hipp_VPTSD.std()), az.hdi(slope_hipp_VPTSD.values.flatten(), hdi_prob=0.89))

HC slope (mean, SD, HDI): -0.004242829701249246 0.028052868123029186 [-0.04869755  0.04170475]
VCC slope (mean, SD, HDI): 0.016159108289039593 0.024453070673778957 [-0.02497877  0.05275106]
VPTSD slope (mean, SD, HDI): 0.033809646158691364 0.02576446244367689 [-0.0083036   0.07272808]


In [29]:
# %% trace
az.summary(trace, var_names=['beta_coupling', 'beta_group_raw', 'beta_interaction_raw'], 
           hdi_prob=0.89)

,mean,sd,hdi_5.5%,hdi_94.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta_coupling,-0.004,0.028,-0.049,0.042,0.001,0.000,3024.0,2805.0,1.0
beta_group_raw[0],0.013,0.016,-0.013,0.039,0.000,0.000,3399.0,3048.0,1.0
beta_group_raw[1],0.011,0.017,-0.015,0.040,0.000,0.000,3522.0,3077.0,1.0
beta_interaction_raw[0],0.020,0.038,-0.035,0.083,0.001,0.001,3105.0,2846.0,1.0
beta_interaction_raw[1],0.038,0.038,-0.022,0.099,0.001,0.001,3201.0,2864.0,1.0
